In [1]:
import numpy as np

# matriz de dissimilaridade do slide 20


labels = ['1', '2', '3', '4', '5', '6']

dist = np.array([
    [0.0,  1.4,  9.7, 15.9, 15.1, 13.7],
    [1.4,  0.0,  9.3, 15.2, 14.4, 12.7],
    [9.7,  9.3,  0.0, 10.9, 10.0, 13.8],
    [15.9, 15.2, 10.9, 0.0,  2.2,  8.2],
    [15.1, 14.4, 10.0, 2.2,  0.0,  8.3],
    [13.7, 12.7, 13.8, 8.2,  8.3,  0.0]
])

In [2]:
# Funções auxiliares

def dissimilaridade_media(obj, grupo):
    """
    Dissimilaridade média de 'obj' em relação aos demais do grupo.
    Usada para encontrar o splinter e decidir quem o segue (slides 7 e 8).
    """
    others = [x for x in grupo if x != obj]
    if not others:
        return 0.0
    return np.mean([dist[obj][x] for x in others])


def gr_splinter(grupo):
    """
    Retorna o objeto mais dissimilar aos demais do grupo.
    Esse objeto inicia o splinter group (slide 7).
    """
    return max(grupo, key=lambda obj: dissimilaridade_media(obj, grupo))


def diametro(grupo):
    """
    Maior distância entre quaisquer dois objetos do grupo.
    Critério para escolher qual cluster dividir (slide 6).
    """
    if len(grupo) <= 1:
        return 0.0
    return max(dist[i][j] for i in grupo for j in grupo if i != j)

In [3]:
# -Algoritmo DIANA ---

def diana():
    """
    Executa o DIANA completo seguindo o pseudocódigo do slide 19.
    A cada passo divide o cluster de maior diâmetro.
    """
    # todos os objetos em um único cluster
    clusters = [list(range(len(labels)))]

    print("Início:", [[labels[i] for i in c] for c in clusters])
    print()

    passo = 1
    while any(len(c) > 1 for c in clusters):

        # escolhemos o cluster com maior diâmetro 
        idx = max(
            [i for i, c in enumerate(clusters) if len(c) > 1],
            key=lambda i: diametro(clusters[i])
        )
        grupo = list(clusters[idx])

        # encontramos o splinter
        splinter = gr_splinter(grupo)
        grupo.remove(splinter)
        splinter_group = [splinter]

        # movemos objetos com diferença positiva para o  grupo splinter
        moveu = True
        while moveu:
            moveu = False
            for obj in list(grupo):
                d_grupo    = dissimilaridade_media(obj, grupo)
                d_splinter = dissimilaridade_media(obj, splinter_group)
                if d_grupo - d_splinter > 0:
                    splinter_group.append(obj)
                    grupo.remove(obj)
                    moveu = True

        # substitupimos o cluster dividido pelos dois novos
        clusters.pop(idx)
        clusters.extend([grupo, splinter_group])

        print(f"Passo {passo}: {[[labels[i] for i in sorted(c)] for c in clusters]}")
        passo += 1

    return clusters


clusters_diana = diana()

Início: [['1', '2', '3', '4', '5', '6']]

Passo 1: [['1', '2', '3'], ['4', '5', '6']]
Passo 2: [['4', '5', '6'], ['1', '2'], ['3']]
Passo 3: [['1', '2'], ['3'], ['4', '5'], ['6']]
Passo 4: [['1', '2'], ['3'], ['6'], ['5'], ['4']]
Passo 5: [['3'], ['6'], ['5'], ['4'], ['2'], ['1']]
